# pvsamlab BESS Sandbox

Demonstrates all four simulation modes after the Phase 3 BESS extension.

| Cell | Mode | PySAM module |
|------|------|--------------|
| 1 | Setup & imports | — |
| 2 | PV-only (`System`) | `Pvsamv1 en_batt=0` |
| 3 | PV + BESS (`PvBessSystem`) | `Pvsamv1 en_batt=1` |
| 4 | BESS-only (`StandaloneBessSystem`) | `StandAloneBattery` |
| 5 | Financial overlay (`compute_lcoe` + `compute_lcos`) | `SingleOwner` / `Cashloan` |

> **Weather data:** Uses cached NSRDB files for Scurry County, TX  
> `lat=33.0278, lon=-100.0814, year='2017'`  
> so no API calls are required to run this notebook.

In [ ]:
import json
import pprint

from pvsamlab import (
    # PV-only
    System,
    TrackingMode,
    # BESS
    Battery,
    BessDispatch,
    PvBessSystem,
    StandaloneBessSystem,
    # Financial
    Financial,
    compute_lcoe,
    compute_lcos,
    compute_npv,
    compute_irr,
)

# ---------------------------------------------------------------------------
# Shared site parameters — reused in every cell below
# ---------------------------------------------------------------------------
SITE_LAT   = 33.0278
SITE_LON   = -100.0814
MET_YEAR   = '2017'           # cached NSRDB file — no download needed
TARGET_MW  = 100_000          # 100 MWac target plant size

print('pvsamlab imported successfully.')

---
## Cell 2 — PV-only (`System`)

Verifies that the existing PV-only path is unchanged after the BESS extension.

In [ ]:
# Build the PV-only system
pv_plant = System(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
)

print(f'System built:  {pv_plant.kwac:,.0f} kWac  |  DC/AC = {pv_plant.dc_ac_ratio:.3f}')

# Run simulation
pv_results = pv_plant.run()

print('\n--- PV-only results ---')
for k, v in pv_results.items():
    print(f'  {k:<50s} {v}')

---
## Cell 3 — PV + BESS (`PvBessSystem`)

Adds a 4-hour LFP battery co-located with the PV plant.  
Uses `Pvsamv1` with `en_batt=1` and **self-consumption** dispatch.

In [ ]:
# Battery: 400 MWh / 100 MW (4-hour duration), LFP, DC-coupled
batt = Battery(
    chemistry='LFP',
    energy_kwh=400_000,       # 400 MWh
    power_kw=100_000,         # 100 MW
    soc_min=10.0,
    soc_max=95.0,
    coupling='DC',
)

# Dispatch: automated self-consumption (charges from excess PV, discharges to load)
disp = BessDispatch(strategy='self_consumption')

# Build PV+BESS system — inherits all PV fields from System
pvbess_plant = PvBessSystem(
    lat=SITE_LAT,
    lon=SITE_LON,
    target_kwac=TARGET_MW,
    target_dcac=1.30,
    met_year=MET_YEAR,
    tracking_mode=TrackingMode.SAT,
    battery=batt,
    dispatch=disp,
)

print(f'PvBessSystem built:  {pvbess_plant.kwac:,.0f} kWac  |  '
      f'Battery: {batt.energy_kwh:,.0f} kWh / {batt.power_kw:,.0f} kW')

# Run simulation
pvbess_results = pvbess_plant.run()

print('\n--- PV+BESS results ---')
for k, v in pvbess_results.items():
    print(f'  {k:<50s} {v}')

---
## Cell 4 — BESS-only (`StandaloneBessSystem`)

No PV.  Simulates a standalone battery against a flat 8760-hour load profile.  
Uses `PySAM.StandAloneBattery`.  Downloads ambient temperature from the same
cached NSRDB file.

In [ ]:
import numpy as np

# Flat load profile: 50 MW constant for all 8760 hours
flat_load_kw = [50_000.0] * 8760

# Battery: 200 MWh / 50 MW (4-hour), self-consumption vs. the flat load
sa_batt = Battery(
    chemistry='LFP',
    energy_kwh=200_000,
    power_kw=50_000,
    soc_min=10.0,
    soc_max=95.0,
)

sa_disp = BessDispatch(strategy='self_consumption')

standalone = StandaloneBessSystem(
    battery=sa_batt,
    dispatch=sa_disp,
    load_profile=flat_load_kw,
    lat=SITE_LAT,
    lon=SITE_LON,
    weather_year=MET_YEAR,
)

print(f'StandaloneBessSystem built:  '
      f'{sa_batt.energy_kwh:,.0f} kWh / {sa_batt.power_kw:,.0f} kW')

# Run simulation (downloads/uses cached ambient temperature)
sa_results = standalone.run()

print('\n--- BESS-only results ---')
for k, v in sa_results.items():
    print(f'  {k:<50s} {v}')

---
## Cell 5 — Financial overlay

Demonstrates `compute_lcoe` (PySAM SingleOwner / Cashloan) and `compute_lcos`
(pure-Python post-simulation) on top of the already-executed simulations.

For the LCOS calculation we construct a simple annual discharge projection:
year-1 discharge from the PV+BESS simulation, degraded by the battery
calendar degradation rate for each subsequent year.

In [ ]:
# ── Financial assumptions ──────────────────────────────────────────────────
fin = Financial(
    analysis_period=25,
    pv_capex_per_kwdc=700.0,      # $/kWdc
    opex_per_kwac_year=15.0,      # $/kWac/yr
    ppa_rate=40.0,                # $/MWh
    ppa_escalation=1.0,           # %/yr
    discount_rate=8.0,            # % real
    inflation_rate=2.5,
    debt_fraction=70.0,
    loan_rate=5.0,
    loan_term=18,
    federal_tax_rate=21.0,
    itc_rate=30.0,
    depreciation_schedule='MACRS5',
)

# ── LCOE / IRR / NPV  (PySAM SingleOwner — kwac > 1000 kW) ────────────────
print('Running compute_lcoe on PV-only system...')
lcoe_pv = compute_lcoe(pv_plant, fin)

print('\n--- PV-only financials (SingleOwner) ---')
for k, v in lcoe_pv.items():
    print(f'  {k:<40s} {v}')

# ── LCOE on PV+BESS system ─────────────────────────────────────────────────
print('\nRunning compute_lcoe on PV+BESS system...')
lcoe_pvbess = compute_lcoe(pvbess_plant, fin)

print('\n--- PV+BESS financials (SingleOwner) ---')
for k, v in lcoe_pvbess.items():
    print(f'  {k:<40s} {v}')

# ── LCOS (post-simulation Python) ─────────────────────────────────────────
# Project annual discharge for 25 years using calendar degradation
y1_discharge_kwh = pvbess_results['batt_annual_discharge_energy_kwh']
deg_rate = batt.calendar_degradation / 100.0   # fraction per year
annual_discharge = [
    y1_discharge_kwh * (1.0 - deg_rate) ** yr
    for yr in range(fin.analysis_period)
]

annual_bess_opex = batt.energy_kwh * batt.opex_per_kwh_year   # $/year
dr = fin.discount_rate / 100.0

lcos = compute_lcos(
    battery=batt,
    annual_discharge_kwh=annual_discharge,
    annual_opex=annual_bess_opex,
    replacement_events=[],   # no replacement assumed
    discount_rate=dr,
)

print(f'\n--- LCOS ---')
print(f'  Year-1 discharge           {y1_discharge_kwh:>15,.1f}  kWh')
print(f'  Annual BESS O&M            {annual_bess_opex:>15,.0f}  $/yr')
print(f'  LCOS                       {lcos:>15.4f}  $/kWh')

# ── Standalone NPV / IRR helpers ──────────────────────────────────────────
# Quick sanity-check using the pure-Python helpers with a toy cash-flow series
example_cashflows = [-1_000_000] + [120_000] * 25    # $1M capex, $120k/yr
ex_npv = compute_npv(example_cashflows, discount_rate=0.08)
ex_irr = compute_irr(example_cashflows)

print(f'\n--- Standalone NPV / IRR (toy example) ---')
print(f'  Cash flows: -$1M upfront, +$120k/yr × 25 years')
print(f'  NPV @ 8%   {ex_npv:>12,.2f}  $')
print(f'  IRR        {ex_irr * 100:>12.3f}  %')